# 05 — Ops Dashboard Queries

This notebook validates the queries that power the **McCain Cold Chain Monitor** AI/BI Dashboard. Run each query to confirm the gold tables produce correct data, then view the actual dashboard artifact.

**Dashboard contents:**
1. KPI row — active shipments, excursions, % excursion rate, total $ at risk
2. Carrier scorecard — excursions per 100 shipments, avg over-tolerance °C
3. Alerts feed — top 20 open excursion events
4. ML vs Rule lead time — bar chart
5. Complaint root-cause mix — stacked bar

> The dashboard is a real Lakeview artifact in this workspace, not just queries in a notebook.

In [0]:
UC_CATALOG = "serverless_stable_qr9if1_catalog"
UC_SCHEMA = "mccain_cold_chain_sju"
spark.sql(f"USE {UC_CATALOG}.{UC_SCHEMA}")

In [0]:
%sql
-- KPI tiles for the dashboard top row
SELECT
  (SELECT count(*) FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.shipments)                                          AS active_shipments,
  (SELECT count(*) FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.gold_excursion_events WHERE severity IN ('MED','HIGH')) AS recent_excursions,
  round(100.0 * (SELECT count(*) FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.silver_shipment_profile WHERE had_excursion) /
                (SELECT count(*) FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.silver_shipment_profile), 1)          AS pct_shipments_excursed,
  coalesce((SELECT sum(dollars_at_risk) FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.gold_excursion_events), 0)     AS total_dollars_at_risk

In [0]:
%sql
-- Latest known position per active trailer, color-coded by worst severity
WITH latest_pos AS (
  SELECT trailer_id, lat, lon, ts,
         row_number() OVER (PARTITION BY trailer_id ORDER BY ts DESC) AS rn
  FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.bronze_gps_pings
),
worst AS (
  SELECT trailer_id,
         max(CASE severity WHEN 'HIGH' THEN 3 WHEN 'MED' THEN 2 WHEN 'LOW' THEN 1 END) AS sev_rank
  FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.gold_excursion_events
  GROUP BY trailer_id
)
SELECT p.trailer_id, p.lat, p.lon, p.ts,
       coalesce(CASE w.sev_rank WHEN 3 THEN 'HIGH' WHEN 2 THEN 'MED' WHEN 1 THEN 'LOW' END, 'OK') AS status
FROM latest_pos p
LEFT JOIN worst w USING (trailer_id)
WHERE rn = 1

In [0]:
%sql
-- Carrier scorecard: excursion rate + dollars at risk per carrier
SELECT
    c.carrier_name,
    count(DISTINCT sp.shipment_id)                                                          AS shipments,
    sum(CASE WHEN sp.had_excursion THEN 1 ELSE 0 END)                                       AS excursions,
    round(100.0 * sum(CASE WHEN sp.had_excursion THEN 1 ELSE 0 END)
                / nullif(count(DISTINCT sp.shipment_id), 0), 1)                             AS excursion_rate_pct,
    round(avg(CASE WHEN sp.had_excursion THEN sp.max_supply_c - sp.upper_tolerance_c END),2) AS avg_over_tolerance_c,
    sum(CASE WHEN e.dollars_at_risk IS NOT NULL THEN e.dollars_at_risk ELSE 0 END)          AS dollars_at_risk
FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.silver_shipment_profile sp
JOIN serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.carriers c ON c.carrier_id = sp.carrier_id
LEFT JOIN serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.gold_excursion_events e ON e.shipment_id = sp.shipment_id
GROUP BY c.carrier_name
ORDER BY excursion_rate_pct DESC

In [0]:
%sql
-- Ops punch-list: top 20 excursion events by severity and dollars at risk
SELECT excursion_id, shipment_id, trailer_id, carrier_id, dest_dc_id,
       sku, start_ts, duration_min, peak_supply_c, peak_over_tolerance_c,
       severity, dollars_at_risk
FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.gold_excursion_events
ORDER BY CASE severity WHEN 'HIGH' THEN 1 WHEN 'MED' THEN 2 ELSE 3 END,
         dollars_at_risk DESC
LIMIT 20

In [0]:
%sql
-- ML vs Rule lead time comparison
WITH first_rule AS (
  SELECT shipment_id, min(detected_ts) AS rule_ts FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.alerts_rule_based GROUP BY shipment_id
),
first_ml AS (
  SELECT shipment_id, min(detected_ts) AS ml_ts FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.alerts_ml_based GROUP BY shipment_id
)
SELECT f.shipment_id,
       cast((unix_timestamp(r.rule_ts) - unix_timestamp(f.ml_ts)) / 60 AS int) AS ml_lead_minutes
FROM first_ml f
JOIN first_rule r ON r.shipment_id = f.shipment_id
ORDER BY ml_lead_minutes DESC

In [0]:
%sql
-- Complaint root-cause mix (stacked bar)
SELECT sensor_root_cause, alleged_cause, count(*) AS complaints
FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.gold_complaint_root_cause
GROUP BY sensor_root_cause, alleged_cause
ORDER BY complaints DESC